In [5]:
"""
Module 1: Data Collection
Fetches spot price, historical prices, option chain, and risk-free rate from Yahoo Finance.
"""
 
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
from typing import Optional
import warnings
warnings.filterwarnings("ignore")
 
 
def fetch_spot_price(ticker: str) -> float:
    t = yf.Ticker(ticker)
    hist = t.history(period="1d")
    if hist.empty:
        raise ValueError(f"No price data found for {ticker}")
    return float(hist["Close"].iloc[-1])
 
 
def fetch_historical_prices(ticker: str, period: str = "1y") -> pd.DataFrame:
    t = yf.Ticker(ticker)
    hist = t.history(period=period)
    hist.index = pd.to_datetime(hist.index).tz_localize(None)
    return hist[["Open", "High", "Low", "Close", "Volume"]].copy()
 
 
def fetch_risk_free_rate() -> float:
    try:
        tnx = yf.Ticker("^IRX")
        hist = tnx.history(period="5d")
        if not hist.empty:
            return float(hist["Close"].iloc[-1]) / 100.0
    except Exception:
        pass
    return 0.05
 
 
def fetch_option_chain(ticker: str, expiry: Optional[str] = None) -> pd.DataFrame:
    t = yf.Ticker(ticker)
    expirations = t.options
    if not expirations:
        raise ValueError(f"No options data for {ticker}")
 
    selected = [expiry] if (expiry and expiry in expirations) else list(expirations[:3])
    spot  = fetch_spot_price(ticker)
    today = datetime.today().date()
    records = []
 
    for exp in selected:
        chain = t.option_chain(exp)
        calls = chain.calls[["strike", "lastPrice", "volume", "openInterest"]].copy()
        puts  = chain.puts[["strike", "lastPrice", "volume", "openInterest"]].copy()
        m = calls.merge(puts, on="strike", suffixes=("_call", "_put"))
        m["date"]       = today
        m["spot"]       = spot
        m["expiry"]     = exp
        m["call_price"] = m["lastPrice_call"]
        m["put_price"]  = m["lastPrice_put"]
        m["volume"]     = m["volume_call"].fillna(0) + m["volume_put"].fillna(0)
        m["oi"]         = m["openInterest_call"].fillna(0) + m["openInterest_put"].fillna(0)
        records.append(m[["date","spot","strike","expiry","call_price","put_price","volume","oi"]])
 
    df = pd.concat(records, ignore_index=True)
    return df.dropna(subset=["call_price","put_price"]).query("call_price>0 and put_price>0")
 
 
def estimate_historical_volatility(ticker: str, window: int = 30) -> float:
    hist = fetch_historical_prices(ticker, period="1y")
    log_ret = np.log(hist["Close"] / hist["Close"].shift(1)).dropna()
    return float(log_ret.rolling(window).std().iloc[-1] * np.sqrt(252))
 
 
def load_market_data(ticker: str) -> dict:
    spot  = fetch_spot_price(ticker)
    rfr   = fetch_risk_free_rate()
    sigma = estimate_historical_volatility(ticker)
    chain = fetch_option_chain(ticker)
    hist  = fetch_historical_prices(ticker)
    return dict(ticker=ticker, spot=spot, rfr=rfr, sigma=sigma,
                chain=chain, history=hist, fetched=datetime.now().isoformat())

In [6]:
"""
Module 2 — Market Parameter Estimation
- Log returns
- Annualized return
- Historical volatility
- Rolling volatility
"""

import numpy as np
import pandas as pd


def calculate_log_returns(price_df):
    """
    Input:
        DataFrame with Close prices

    Returns:
        Series of log returns
    """
    close = price_df["Close"]
    returns = np.log(close / close.shift(1))
    return returns.dropna()


def annualized_return(price_df):
    r = calculate_log_returns(price_df)
    return float(r.mean() * 252)


def annualized_volatility(price_df):
    r = calculate_log_returns(price_df)
    return float(r.std() * np.sqrt(252))


def rolling_volatility(price_df, window=30):
    r = calculate_log_returns(price_df)
    vol = r.rolling(window).std() * np.sqrt(252)
    return vol.dropna()


def estimate_market_parameters(price_df):
    return {
        "annual_return": annualized_return(price_df),
        "annual_volatility": annualized_volatility(price_df),
        "rolling_volatility": rolling_volatility(price_df)
    }

In [7]:
"""
Module 3- Geometric Brownian Motion simulation engine.
dS = mu*S*dt + sigma*S*dW
"""
 
import numpy as np
from typing import Tuple
 
 
def simulate_gbm_paths(
    S0: float,
    mu: float,
    sigma: float,
    T: float,
    n_steps: int,
    n_paths: int,
    seed: int = 42,
    antithetic: bool = True,
) -> np.ndarray:
    """
    Simulate GBM paths.
 
    Returns
    -------
    paths : ndarray of shape (n_paths, n_steps+1)
    """
    rng = np.random.default_rng(seed)
    dt  = T / n_steps
 
    if antithetic:
        half = n_paths // 2
        z    = rng.standard_normal((half, n_steps))
        z    = np.vstack([z, -z])
    else:
        z = rng.standard_normal((n_paths, n_steps))
 
    increments = (mu - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * z
    log_paths  = np.concatenate(
        [np.zeros((n_paths, 1)), np.cumsum(increments, axis=1)], axis=1
    )
    return S0 * np.exp(log_paths)
 
 
def simulate_gbm_terminal(
    S0: float,
    mu: float,
    sigma: float,
    T: float,
    n_paths: int,
    seed: int = 42,
) -> np.ndarray:
    """Return only terminal prices (faster for European / vanilla)."""
    rng = np.random.default_rng(seed)
    z   = rng.standard_normal(n_paths)
    return S0 * np.exp((mu - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * z)
 
 
def time_grid(T: float, n_steps: int) -> np.ndarray:
    return np.linspace(0, T, n_steps + 1)

In [12]:
"""
Module 4 — European Option Pricing
Methods: Black-Scholes, Binomial Tree, Monte Carlo
"""
 
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
 
# ─────────────────────────────────────────────
# Black-Scholes
# ─────────────────────────────────────────────
 
def bs_d1_d2(S, K, T, r, sigma, q=0.0):
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return d1, d2
 
 
def black_scholes(S: float, K: float, T: float, r: float, sigma: float,
                  option_type: str = "call", q: float = 0.0) -> float:
    """Closed-form Black-Scholes price."""
    if T <= 0:
        return max(S - K, 0) if option_type == "call" else max(K - S, 0)
    d1, d2 = bs_d1_d2(S, K, T, r, sigma, q)
    if option_type == "call":
        return S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)
 
 
# ─────────────────────────────────────────────
# Binomial Tree (CRR)
# ─────────────────────────────────────────────
 
def binomial_european(S: float, K: float, T: float, r: float, sigma: float,
                      option_type: str = "call", n: int = 200) -> float:
    dt = T / n
    u  = np.exp(sigma * np.sqrt(dt))
    d  = 1 / u
    p  = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
 
    # Terminal payoffs
    j   = np.arange(n + 1)
    ST  = S * u ** (n - 2 * j)
    if option_type == "call":
        V = np.maximum(ST - K, 0)
    else:
        V = np.maximum(K - ST, 0)
 
    # Backward induction
    for _ in range(n):
        V = disc * (p * V[:-1] + (1 - p) * V[1:])
    return float(V[0])
 
 
# ─────────────────────────────────────────────
# Monte Carlo
# ─────────────────────────────────────────────
 
def monte_carlo_european(S: float, K: float, T: float, r: float, sigma: float,
                         option_type: str = "call", n_paths: int = 50_000,
                         seed: int = 42) -> dict:
    ST = simulate_gbm_terminal(S, r, sigma, T, n_paths, seed)
    if option_type == "call":
        payoffs = np.maximum(ST - K, 0)
    else:
        payoffs = np.maximum(K - ST, 0)
 
    disc_payoffs = np.exp(-r * T) * payoffs
    price = disc_payoffs.mean()
    se    = disc_payoffs.std() / np.sqrt(n_paths)
    return {"price": float(price), "se": float(se),
            "ci_low": float(price - 1.96*se), "ci_high": float(price + 1.96*se)}
 
 
def price_european(S, K, T, r, sigma, option_type="call", method="black_scholes",
                   n_paths=50_000, n_steps=200):
    method = method.lower().replace(" ", "_").replace("-", "_")
    if method == "black_scholes":
        return {"price": black_scholes(S, K, T, r, sigma, option_type), "method": "Black-Scholes"}
    elif method == "binomial":
        return {"price": binomial_european(S, K, T, r, sigma, option_type, n_steps), "method": "Binomial"}
    elif method == "monte_carlo":
        res = monte_carlo_european(S, K, T, r, sigma, option_type, n_paths)
        res["method"] = "Monte Carlo"
        return res
    else:
        raise ValueError(f"Unknown method: {method}")

In [9]:
"""
Module 5 — American Option Pricing
Methods: Binomial Tree with early exercise
"""
 
import numpy as np
 
 
def binomial_american(S: float, K: float, T: float, r: float, sigma: float,
                      option_type: str = "put", n: int = 300) -> dict:
    """
    CRR binomial tree with early-exercise check at every node.
    Returns price + early-exercise boundary.
    """
    dt  = T / n
    u   = np.exp(sigma * np.sqrt(dt))
    d   = 1.0 / u
    p   = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
 
    # Terminal stock prices
    j  = np.arange(n + 1)
    ST = S * (u ** (n - j)) * (d ** j)
 
    if option_type == "call":
        V = np.maximum(ST - K, 0.0)
        intrinsic_fn = lambda s: np.maximum(s - K, 0.0)
    else:
        V = np.maximum(K - ST, 0.0)
        intrinsic_fn = lambda s: np.maximum(K - s, 0.0)
 
    early_exercise_boundary = []
 
    for i in range(n - 1, -1, -1):
        j_i = np.arange(i + 1)
        S_i = S * (u ** (i - j_i)) * (d ** j_i)
        hold = disc * (p * V[:i+1] + (1 - p) * V[1:i+2])
        exer = intrinsic_fn(S_i)
        V    = np.maximum(hold, exer)
 
        # record boundary: lowest S where early exercise is optimal
        exercise_mask = exer > hold
        if exercise_mask.any():
            boundary_S = S_i[exercise_mask].max()
        else:
            boundary_S = np.nan
        early_exercise_boundary.append((i * dt, boundary_S))
 
    early_exercise_boundary.reverse()
    return {
        "price":    float(V[0]),
        "method":   "Binomial American",
        "boundary": early_exercise_boundary,
    }
 
 
def early_exercise_premium(S, K, T, r, sigma, option_type="put", n=300):
    """American price minus European price = early-exercise premium."""
    from models.european import binomial_european
    am = binomial_american(S, K, T, r, sigma, option_type, n)["price"]
    eu = binomial_european(S, K, T, r, sigma, option_type, n)
    return {"american": am, "european": eu, "premium": am - eu}
 
 
def price_american(S, K, T, r, sigma, option_type="put", n_steps=300):
    return binomial_american(S, K, T, r, sigma, option_type, n_steps)

In [13]:
"""
Module 6 — Asian Option Pricing
Method: Monte Carlo (arithmetic & geometric average)
"""
 
import numpy as np
 
 
def monte_carlo_asian(
    S: float, K: float, T: float, r: float, sigma: float,
    option_type: str = "call",
    average_type: str = "arithmetic",   # "arithmetic" | "geometric"
    n_paths: int = 50_000,
    n_steps: int = 252,
    seed: int = 42,
) -> dict:
    paths = simulate_gbm_paths(S, r, sigma, T, n_steps, n_paths, seed)
    # paths shape: (n_paths, n_steps+1); exclude S0
    prices = paths[:, 1:]
 
    if average_type == "arithmetic":
        avg = prices.mean(axis=1)
    else:
        avg = np.exp(np.log(prices).mean(axis=1))
 
    if option_type == "call":
        payoffs = np.maximum(avg - K, 0)
    else:
        payoffs = np.maximum(K - avg, 0)
 
    disc_payoffs = np.exp(-r * T) * payoffs
    price = disc_payoffs.mean()
    se    = disc_payoffs.std() / np.sqrt(n_paths)
 
    return {
        "price":       float(price),
        "se":          float(se),
        "ci_low":      float(price - 1.96 * se),
        "ci_high":     float(price + 1.96 * se),
        "average_type": average_type,
        "method":      "Monte Carlo Asian",
    }
 
 
def price_asian(S, K, T, r, sigma, option_type="call",
                average_type="arithmetic", n_paths=50_000, n_steps=252):
    return monte_carlo_asian(S, K, T, r, sigma, option_type,
                             average_type, n_paths, n_steps)

In [14]:
"""
Module 7 — Barrier Option Pricing
Method: Monte Carlo
Supported types: up-and-out, up-and-in, down-and-out, down-and-in
"""
 
import numpy as np
 
 
def monte_carlo_barrier(
    S: float, K: float, T: float, r: float, sigma: float,
    barrier: float,
    barrier_type: str = "down-and-out",   # up/down + in/out
    option_type: str  = "call",
    n_paths: int = 50_000,
    n_steps: int = 252,
    rebate: float = 0.0,
    seed: int = 42,
) -> dict:
    paths   = simulate_gbm_paths(S, r, sigma, T, n_steps, n_paths, seed)
    barrier_type = barrier_type.lower()
 
    direction, knock = barrier_type.split("-and-")
 
    if direction == "up":
        crossed = (paths.max(axis=1) >= barrier)
    else:  # down
        crossed = (paths.min(axis=1) <= barrier)
 
    ST = paths[:, -1]
    if option_type == "call":
        base_payoff = np.maximum(ST - K, 0)
    else:
        base_payoff = np.maximum(K - ST, 0)
 
    if knock == "out":
        payoffs = np.where(crossed, rebate, base_payoff)
    else:  # in
        payoffs = np.where(crossed, base_payoff, rebate)
 
    disc_payoffs = np.exp(-r * T) * payoffs
    price = disc_payoffs.mean()
    se    = disc_payoffs.std() / np.sqrt(n_paths)
 
    return {
        "price":        float(price),
        "se":           float(se),
        "ci_low":       float(price - 1.96 * se),
        "ci_high":      float(price + 1.96 * se),
        "barrier_type": barrier_type,
        "barrier":      barrier,
        "method":       "Monte Carlo Barrier",
    }
 
 
def price_barrier(S, K, T, r, sigma, barrier, barrier_type="down-and-out",
                  option_type="call", n_paths=50_000, n_steps=252, rebate=0.0):
    return monte_carlo_barrier(S, K, T, r, sigma, barrier, barrier_type,
                               option_type, n_paths, n_steps, rebate)

In [16]:
"""
Module 8 — Greeks
Analytic (Black-Scholes) + finite difference for exotic options.
Covers: Delta, Gamma, Vega, Theta, Rho
"""
 
import numpy as np
from scipy.stats import norm
 
 
# ─────────────────────────────────────────────
# Analytic Greeks (Black-Scholes)
# ─────────────────────────────────────────────
 
def bs_greeks(S: float, K: float, T: float, r: float, sigma: float,
              option_type: str = "call", q: float = 0.0) -> dict:
    if T <= 0:
        return dict(delta=0, gamma=0, vega=0, theta=0, rho=0)
 
    d1, d2 = bs_d1_d2(S, K, T, r, sigma, q)
    Nd1  = norm.cdf(d1)
    Nd2  = norm.cdf(d2)
    nd1  = norm.pdf(d1)
 
    if option_type == "call":
        delta = np.exp(-q * T) * Nd1
        theta = (
            -np.exp(-q * T) * S * nd1 * sigma / (2 * np.sqrt(T))
            - r * K * np.exp(-r * T) * Nd2
            + q * S * np.exp(-q * T) * Nd1
        ) / 365
        rho = K * T * np.exp(-r * T) * Nd2 / 100
    else:
        delta = -np.exp(-q * T) * norm.cdf(-d1)
        theta = (
            -np.exp(-q * T) * S * nd1 * sigma / (2 * np.sqrt(T))
            + r * K * np.exp(-r * T) * norm.cdf(-d2)
            - q * S * np.exp(-q * T) * norm.cdf(-d1)
        ) / 365
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100
 
    gamma = np.exp(-q * T) * nd1 / (S * sigma * np.sqrt(T))
    vega  = S * np.exp(-q * T) * nd1 * np.sqrt(T) / 100
 
    return dict(delta=float(delta), gamma=float(gamma),
                vega=float(vega), theta=float(theta), rho=float(rho))
 
 
# ─────────────────────────────────────────────
# Finite-difference Greeks (generic pricer)
# ─────────────────────────────────────────────
 
def fd_greeks(pricer_fn, S, K, T, r, sigma, option_type="call", **kwargs) -> dict:
    """
    pricer_fn(S, K, T, r, sigma, option_type, **kwargs) -> dict with 'price' key.
    Uses central differences.
    """
    dS    = S * 0.01
    dsig  = 0.001
    dt    = 1 / 365
    dr    = 0.0001
 
    p0    = pricer_fn(S,      K, T,      r,    sigma,       option_type, **kwargs)["price"]
    p_up  = pricer_fn(S+dS,   K, T,      r,    sigma,       option_type, **kwargs)["price"]
    p_dn  = pricer_fn(S-dS,   K, T,      r,    sigma,       option_type, **kwargs)["price"]
    p_sig = pricer_fn(S,      K, T,      r,    sigma+dsig,  option_type, **kwargs)["price"]
    p_t   = pricer_fn(S,      K, T-dt,   r,    sigma,       option_type, **kwargs)["price"] if T > dt else p0
    p_r   = pricer_fn(S,      K, T,      r+dr, sigma,       option_type, **kwargs)["price"]
 
    delta = (p_up - p_dn) / (2 * dS)
    gamma = (p_up - 2*p0 + p_dn) / (dS ** 2)
    vega  = (p_sig - p0) / dsig / 100
    theta = (p_t - p0) / dt / 365
    rho   = (p_r - p0) / dr / 100
 
    return dict(delta=float(delta), gamma=float(gamma),
                vega=float(vega), theta=float(theta), rho=float(rho))
 
 
def compute_greeks(S, K, T, r, sigma, option_type="call",
                   method="analytic", pricer_fn=None, **kwargs):
    if method == "analytic":
        return bs_greeks(S, K, T, r, sigma, option_type)
    elif method == "finite_difference" and pricer_fn is not None:
        return fd_greeks(pricer_fn, S, K, T, r, sigma, option_type, **kwargs)
    else:
        raise ValueError("method must be 'analytic' or 'finite_difference' with pricer_fn")

In [17]:
"""
Module 9 — Validation
- BS vs MC vs Binomial
- Convergence studies
- Put-Call Parity
- Barrier sanity checks
"""

import numpy as np
import pandas as pd

# A. Method comparison

def compare_european_methods(S,K,T,r,sigma,option_type="call"):
    bs = black_scholes(S,K,T,r,sigma,option_type)
    mc = monte_carlo_european(S,K,T,r,sigma,option_type)["price"]
    bt = binomial_european(S,K,T,r,sigma,option_type)

    return pd.DataFrame({"Method":["Black-Scholes", "Monte Carlo", "Binomial"],"Price": [bs, mc, bt]})


# B. Monte Carlo Convergence

def monte_carlo_convergence(S,K,T,r,sigma,option_type="call",paths=[1000,5000,10000,50000,100000]):
    results = []

    for p in paths:
        price = monte_carlo_european(S,K,T,r,sigma,option_type,n_paths=p)["price"]
        results.append({"paths": p,"price": price})

    return pd.DataFrame(results)

# C. Binomial convergence

def binomial_convergence(S,K,T,r,sigma,option_type="call",nodes=[50,100,250,500,1000]):
    results = []

    for n in nodes:
        price = binomial_european(S,K,T,r,sigma,option_type,n=n)

        results.append({"nodes": n,"price": price})

    return pd.DataFrame(results)


# D. Put-call parity

def put_call_parity(S,K,T,r,sigma):
    C = black_scholes(S,K,T,r,sigma,"call")

    P = black_scholes(S,K,T,r,sigma,"put")

    lhs = C - P
    rhs = S - K*np.exp(-r*T)

    return {"lhs": lhs,
        "rhs": rhs,
        "difference": abs(lhs-rhs)}


# E. Barrier check
def barrier_vanilla_check(S,K,T,r,sigma,option_type="call"):

    vanilla = black_scholes(S,K,T,r,sigma,option_type)

    barrier = monte_carlo_barrier(S,K,T,r,sigma,barrier=1e10,barrier_type="up-and-out",option_type=option_type)["price"]

    return {"vanilla_price": vanilla,
        "barrier_price": barrier,
        "difference": abs(vanilla-barrier)}

In [19]:
"""
Module 10 — Calibration
- Implied volatility surface extraction from option chain
- Model vs market pricing error (RMSE, MAE, MAPE)
- SVI (Stochastic Volatility Inspired) smile fitting
"""
 
import numpy as np
import pandas as pd
from scipy.optimize import minimize, brentq
from typing import Literal
 
 
# ─────────────────────────────────────────────
# Implied Volatility Surface
# ─────────────────────────────────────────────
 
def build_iv_surface(chain_df: pd.DataFrame, S: float, r: float) -> pd.DataFrame:
    """
    Compute implied vol for each row in the option chain DataFrame.
    Expects columns: strike, T, call_price / put_price, option_type
    """
    rows = []
    for _, row in chain_df.iterrows():
        K = row["strike"]
        T = row["T"]
        otype = row["option_type"]
        mkt_price = row["call_price"] if otype == "call" else row["put_price"]
 
        if pd.isna(mkt_price) or mkt_price <= 0 or T <= 0:
            continue
 
        iv = implied_volatility(mkt_price, S, K, r, T, otype)
        if np.isnan(iv) or iv <= 0:
            continue
 
        rows.append({
            "strike": K,
            "T": round(T, 4),
            "expiry": row.get("expiry", ""),
            "option_type": otype,
            "market_price": mkt_price,
            "implied_vol": iv,
            "moneyness": np.log(K / S),
        })
 
    return pd.DataFrame(rows)
 
 
# ─────────────────────────────────────────────
# Model vs Market Error
# ─────────────────────────────────────────────
 
def model_vs_market(
    iv_surface: pd.DataFrame, S: float, r: float, sigma_model: float
) -> pd.DataFrame:
    """
    Compare BS model prices (using flat vol) against market prices.
    Returns per-row errors and summary statistics.
    """
    records = []
    for _, row in iv_surface.iterrows():
        model_price = black_scholes(
            S, row["strike"], r, sigma_model, row["T"], row["option_type"]
        )
        mkt = row["market_price"]
        error = model_price - mkt
        pct_error = (error / mkt * 100) if mkt > 0 else np.nan
        records.append({
            **row.to_dict(),
            "model_price": model_price,
            "error": error,
            "abs_error": abs(error),
            "pct_error": pct_error,
        })
 
    df = pd.DataFrame(records)
    return df
 
 
def pricing_errors(model_vs_mkt_df: pd.DataFrame) -> dict:
    df = model_vs_mkt_df.dropna(subset=["error", "pct_error"])
    if df.empty:
        return {}
    return {
        "RMSE": float(np.sqrt((df["error"] ** 2).mean())),
        "MAE": float(df["abs_error"].mean()),
        "MAPE": float(df["pct_error"].abs().mean()),
        "bias": float(df["error"].mean()),
        "n_options": len(df),
    }
 
 
# ─────────────────────────────────────────────
# SVI Smile Fit (per expiry)
# ─────────────────────────────────────────────
 
def svi_variance(k, a, b, rho, m, sigma_svi):
    """SVI raw parametrization: w(k) = a + b*(rho*(k-m) + sqrt((k-m)^2 + sigma^2))"""
    return a + b * (rho * (k - m) + np.sqrt((k - m) ** 2 + sigma_svi ** 2))
 
 
def fit_svi(iv_surface_slice: pd.DataFrame) -> dict:
    """
    Fit SVI smile to a single expiry slice.
    iv_surface_slice must have 'moneyness' (log(K/S)) and 'implied_vol' columns.
    """
    k = iv_surface_slice["moneyness"].values
    w_mkt = iv_surface_slice["implied_vol"].values ** 2 * iv_surface_slice["T"].values
 
    def obj(params):
        a, b, rho, m, sig = params
        if b < 0 or sig <= 0 or abs(rho) >= 1:
            return 1e10
        w_fit = svi_variance(k, a, b, rho, m, sig)
        if (w_fit < 0).any():
            return 1e10
        return np.sum((w_fit - w_mkt) ** 2)
 
    x0 = [0.04, 0.1, -0.3, 0.0, 0.1]
    bounds = [(1e-6, 1), (1e-6, 2), (-0.999, 0.999), (-1, 1), (1e-4, 2)]
    res = minimize(obj, x0, method="L-BFGS-B", bounds=bounds)
 
    a, b, rho, m, sig = res.x
    T_avg = iv_surface_slice["T"].mean()
    k_grid = np.linspace(k.min() - 0.1, k.max() + 0.1, 100)
    w_grid = svi_variance(k_grid, a, b, rho, m, sig)
    iv_grid = np.sqrt(np.maximum(w_grid / T_avg, 0))
 
    return {
        "params": {"a": a, "b": b, "rho": rho, "m": m, "sigma": sig},
        "success": res.success,
        "k_grid": k_grid.tolist(),
        "iv_grid": iv_grid.tolist(),
    }
 
 
# ─────────────────────────────────────────────
# Calibrate flat vol to minimize RMSE
# ─────────────────────────────────────────────
 
def calibrate_flat_vol(chain_df: pd.DataFrame, S: float, r: float) -> dict:
    """Find the single flat vol that minimizes RMSE against market prices."""
 
    def total_rmse(sigma_vec):
        sigma = sigma_vec[0]
        errors = []
        for _, row in chain_df.iterrows():
            K, T = row["strike"], row["T"]
            otype = row["option_type"]
            mkt = row["call_price"] if otype == "call" else row["put_price"]
            if pd.isna(mkt) or mkt <= 0 or T <= 0:
                continue
            model = black_scholes(S, K, r, sigma, T, otype)
            errors.append((model - mkt) ** 2)
        return np.sqrt(np.mean(errors)) if errors else 1e10
 
    res = minimize(total_rmse, [0.2], method="Nelder-Mead",
                   options={"xatol": 1e-6, "fatol": 1e-8})
    return {"calibrated_vol": float(res.x[0]), "rmse": float(res.fun)}

In [21]:
"""
Module 11 — Visualization
"""

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np


# ==========================================================
# 1. GBM Paths
# ==========================================================

def plot_gbm_paths(paths,
                   n_show=20):

    plt.figure(figsize=(10,6))

    for i in range(
            min(n_show,
                paths.shape[0])):
        plt.plot(paths[i])

    plt.title(
        "Geometric Brownian Motion Paths"
    )
    plt.xlabel("Time Step")
    plt.ylabel("Stock Price")
    plt.grid(True)
    plt.show()


# ==========================================================
# 2. Monte Carlo Convergence
# ==========================================================

def plot_mc_convergence(df):

    plt.figure(figsize=(8,5))

    plt.plot(
        df["Paths"],
        df["Price"],
        marker="o"
    )

    plt.xscale("log")

    plt.xlabel(
        "Number of Simulation Paths"
    )

    plt.ylabel(
        "Option Price"
    )

    plt.title(
        "Monte Carlo Convergence"
    )

    plt.grid(True)
    plt.show()


# ==========================================================
# 3. Binomial Convergence
# ==========================================================

def plot_binomial_convergence(df):

    plt.figure(figsize=(8,5))

    plt.plot(
        df["Nodes"],
        df["Price"],
        marker="o"
    )

    plt.xlabel(
        "Number of Tree Nodes"
    )

    plt.ylabel(
        "Option Price"
    )

    plt.title(
        "Binomial Tree Convergence"
    )

    plt.grid(True)
    plt.show()


# ==========================================================
# 4. Greeks Sensitivity
# ==========================================================

def plot_greek(
        x,
        y,
        greek_name):

    plt.figure(figsize=(8,5))

    plt.plot(
        x,
        y
    )

    plt.xlabel(
        "Underlying Price"
    )

    plt.ylabel(
        greek_name
    )

    plt.title(
        f"{greek_name} Sensitivity"
    )

    plt.grid(True)
    plt.show()


# ==========================================================
# 5. Market vs Model
# ==========================================================

def plot_market_vs_model(df):

    plt.figure(figsize=(7,7))

    plt.scatter(
        df["market_price"],
        df["model_price"]
    )

    mn = min(df["market_price"])
    mx = max(df["market_price"])

    plt.plot(
        [mn,mx],
        [mn,mx]
    )

    plt.xlabel(
        "Market Price"
    )

    plt.ylabel(
        "Model Price"
    )

    plt.title(
        "Market vs Model Prices"
    )

    plt.grid(True)
    plt.show()


# ==========================================================
# 6. Implied Volatility Surface
# ==========================================================

def plot_iv_surface(iv_df):

    fig = plt.figure(
        figsize=(10,7)
    )

    ax = fig.add_subplot(
        111,
        projection="3d"
    )

    ax.scatter(
        iv_df["strike"],
        iv_df["T"],
        iv_df["implied_vol"]
    )

    ax.set_xlabel(
        "Strike"
    )

    ax.set_ylabel(
        "Time to Maturity"
    )

    ax.set_zlabel(
        "Implied Volatility"
    )

    plt.title(
        "Implied Volatility Surface"
    )

    plt.show()